# Jailbreak likelihood classifier — issue #5
A standalone, CPU-only notebook. Run all cells from the repository root. No external training data, model downloads, or companion Python modules. Prompts are **untrusted data**, never executable instructions.

Fixed equal-weight ensemble of word/character TF-IDF logistic models and calibrated linear SVM models. All predictive feature fitting is inside each training fold. Near-duplicate groups are kept together. TARGET is a model-estimated jailbreak likelihood, not a calibrated security guarantee; threshold 0.5 means jailbreak. The latest requested public ROC-AUC target is 0.97, **not a guaranteed result**.

Research before implementation: [issue and comments](https://github.com/johndesantis/modeltrain/issues/5), [scikit-learn text pipeline example](https://scikit-learn.org/stable/auto_examples/model_selection/plot_grid_search_text_feature_extraction.html).

In [1]:
from pathlib import Path
import csv, hashlib, json, platform, re, time, unicodedata, warnings
import numpy as np
import pandas as pd
import scipy, sklearn
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, brier_score_loss
from sklearn.neighbors import NearestNeighbors
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits
warnings.filterwarnings('error', category=ConvergenceWarning)
SEED = 42
FOLDS = 5
NEAR_DUPLICATE_COSINE = 0.80
ROOT = Path.cwd()
START = time.monotonic()
# Avoid oversubscribed BLAS on CPU environments.
thread_limit = threadpool_limits(limits=2)
print('Python', platform.python_version(), '| sklearn', sklearn.__version__)

Python 3.11.2 | sklearn 1.7.2


## 1. Strict CSV validation (IDs are never predictive features)

In [2]:
def require(condition, message):
    if not condition:
        raise ValueError(message)

def normalize(text):
    text = unicodedata.normalize('NFKC', text).casefold()
    text = ''.join(c for c in text if unicodedata.category(c) != 'Cf')
    return ' '.join(text.split())

def validate_frame(frame, columns):
    require(list(frame.columns) == columns, f'Expected columns {columns}')
    require(len(frame) > 0, 'Empty dataset')
    require(not frame.isna().any().any(), 'Missing values')
    require(frame['Id'].str.strip().ne('').all(), 'Blank IDs')
    require(frame['Id'].is_unique, 'Duplicate IDs')
    if 'text' in columns:
        require(frame.text.map(normalize).ne('').all(), 'Empty normalized prompts')
    if 'label' in columns:
        require(set(frame.label) == {'benign', 'jailbreak'}, 'Expected both binary labels')
    return frame

def load_csv(name, columns):
    path = ROOT / name
    with path.open(newline='', encoding='utf-8') as f:
        rows = csv.reader(f)
        require(next(rows, None) == columns, f'{name}: invalid header')
        require(all(len(row) == len(columns) for row in rows), f'{name}: malformed CSV row')
    return validate_frame(pd.read_csv(path, dtype=str, keep_default_na=False), columns)

train = load_csv('train.csv', ['Id', 'text', 'label'])
test = load_csv('test.csv', ['Id', 'text'])
sample = load_csv('sample_submission.csv', ['Id', 'TARGET'])
require(set(train.Id).isdisjoint(test.Id), 'Train/test ID overlap')
require(sample.Id.tolist() == test.Id.tolist(), 'Sample/test ID order mismatch')
train['normalized'] = train.text.map(normalize)
test['normalized'] = test.text.map(normalize)
require(set(train.normalized).isdisjoint(test.normalized), 'Normalized train/test text overlap')
conflicts = train.groupby('normalized').label.nunique()
require(conflicts.max() == 1, 'Conflicting labels for identical normalized prompts')
duplicate_rows = int(train.normalized.duplicated().sum())
train = train.drop_duplicates('normalized').reset_index(drop=True)
texts = train.normalized.to_numpy()
y = train.label.map({'benign': 0, 'jailbreak': 1}).to_numpy()
print('Training:', len(train), '| test:', len(test), '| removed duplicates:', duplicate_rows)
print(train.label.value_counts().to_dict())

Training: 4000 | test: 1000 | removed duplicates: 0
{'benign': 2401, 'jailbreak': 1599}


## 2. Conservative near-duplicate grouping
Unsupervised hashing is used **only** for split grouping, never as predictive features. Connected components keep transitive template families in one fold. This is a lexical safeguard, not a semantic/source-independence guarantee.

In [3]:
from sklearn.feature_extraction.text import HashingVectorizer
# No learned vocabulary or IDF, no test prompts and no labels used for grouping.
group_vectorizer = HashingVectorizer(analyzer='char_wb', ngram_range=(3, 5),
                                    n_features=2**18, alternate_sign=False, norm='l2')
group_matrix = group_vectorizer.transform(texts)
neighbor_index = NearestNeighbors(metric='cosine', algorithm='brute', n_jobs=2).fit(group_matrix)
graph = neighbor_index.radius_neighbors_graph(group_matrix,
    radius=1 - NEAR_DUPLICATE_COSINE + 1e-12, mode='connectivity')
_, groups = connected_components(graph, directed=False)
splits = list(StratifiedGroupKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
              .split(texts, y, groups))
for tr, va in splits:
    require(set(groups[tr]).isdisjoint(groups[va]), 'Group leakage')
    require(len(np.unique(y[tr])) == len(np.unique(y[va])) == 2, 'Single-class fold')
print('Groups:', len(np.unique(groups)), '| largest:', int(np.bincount(groups).max()))
print('Validation fold sizes:', [len(va) for _, va in splits])

Groups: 3855 | largest: 85
Validation fold sizes: [729, 803, 793, 791, 884]


## 3. Model definitions
SVM calibration uses three inner training splits; outer validation remains untouched by fitting. Fixed equal blending avoids weight optimization. The initial NB-weighted ensemble scored 0.94881 grouped OOF ROC-AUC (word LR 0.94375, char LR 0.95412); NB members lowered performance and were replaced with SVMs. This model-family comparison uses the same CV, so the reported result is a development estimate, not an untouched final holdout. Inner calibration shares an outer-training vocabulary and is not itself a leakage-free performance estimate. No IDs, sample targets, test labels, or test-fitted vocabulary enter training.

In [4]:
MEMBERS = ['word_lr', 'word_svm', 'char_lr', 'char_svm']

def vectorizer(view):
    common = dict(sublinear_tf=True, min_df=2, max_df=1.0, dtype=np.float64)
    if view == 'word':
        return TfidfVectorizer(ngram_range=(1, 2), token_pattern=r'(?u)\b\w+\b',
                               max_features=120000, **common)
    return TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 6),
                           max_features=180000, **common)

def fit_view(train_text, labels, prediction_text, view):
    vect = vectorizer(view)
    X = vect.fit_transform(train_text)
    Z = vect.transform(prediction_text)
    outputs, fitted = [], []
    for use_svm in (False, True):
        ratio = np.ones(X.shape[1])
        if use_svm:
            model = CalibratedClassifierCV(
                LinearSVC(C=0.7, dual='auto', max_iter=10000, random_state=SEED),
                method='sigmoid', cv=3, ensemble=False)
        else:
            model = LogisticRegression(C=8.0, solver='liblinear',
                                       max_iter=2000, random_state=SEED)
        model.fit(X.multiply(ratio).tocsr(), labels)
        require(model.classes_.tolist() == [0, 1], 'Incorrect class direction')
        outputs.append(model.predict_proba(Z.multiply(ratio).tocsr())[:, 1])
        fitted.append((model, ratio))
    return np.column_stack(outputs), (vect, fitted)

## 4. Five-fold out-of-fold evaluation (no test feedback)

In [5]:
oof = np.full((len(train), len(MEMBERS)), np.nan)
fold_assignment = np.full(len(train), -1)
fold_metrics = []
for fold, (tr, va) in enumerate(splits):
    for offset, view in [(0, 'word'), (2, 'char')]:
        predictions, _ = fit_view(texts[tr], y[tr], texts[va], view)
        oof[va, offset:offset+2] = predictions
    fold_assignment[va] = fold
    auc = roc_auc_score(y[va], oof[va].mean(axis=1))
    fold_metrics.append(float(auc))
    print(f'Fold {fold+1}: ROC-AUC={auc:.6f}', flush=True)
require(np.isfinite(oof).all(), 'Incomplete OOF predictions')
oof_score = oof.mean(axis=1)
metrics = {name: float(roc_auc_score(y, oof[:, i])) for i, name in enumerate(MEMBERS)}
metrics['ensemble'] = float(roc_auc_score(y, oof_score))
print('OOF ROC-AUC:', json.dumps(metrics, indent=2))
print('Accuracy:', accuracy_score(y, oof_score >= .5),
      '| F1:', f1_score(y, oof_score >= .5), '| Brier:', brier_score_loss(y, oof_score))

Fold 1: ROC-AUC=0.957346


Fold 2: ROC-AUC=0.968842


Fold 3: ROC-AUC=0.958368


Fold 4: ROC-AUC=0.952899


Fold 5: ROC-AUC=0.955757


OOF ROC-AUC: {
  "word_lr": 0.943751548174502,
  "word_svm": 0.9498358902468977,
  "char_lr": 0.9541167311202153,
  "char_svm": 0.9570459879782216,
  "ensemble": 0.9576109495756797
}
Accuracy: 0.88875 | F1: 0.8595771536762385 | Brier: 0.08155728712934916


## 5. Group bootstrap uncertainty
Resampling whole groups accounts for lexical dependence, conditional on these fitted OOF predictions. It does not include model-selection/training uncertainty and is not a public-score interval.

In [6]:
rng = np.random.default_rng(SEED)
group_rows = [np.flatnonzero(groups == g) for g in np.unique(groups)]
bootstrap_auc = []
for _ in range(500):
    idx = np.concatenate([group_rows[g] for g in rng.integers(0, len(group_rows), len(group_rows))])
    if len(np.unique(y[idx])) == 2:
        bootstrap_auc.append(roc_auc_score(y[idx], oof_score[idx]))
auc_interval = np.quantile(bootstrap_auc, [.025, .975]).tolist()
print('Group-bootstrap 95% interval:', auc_interval)

Group-bootstrap 95% interval: [0.9477898427470931, 0.9660531433192228]


## 6. Refit on all labeled rows and score untouched test prompts

In [7]:
test_members = np.empty((len(test), len(MEMBERS)))
models = {}
for offset, view in [(0, 'word'), (2, 'char')]:
    predictions, models[view] = fit_view(texts, y, test.normalized.to_numpy(), view)
    test_members[:, offset:offset+2] = predictions
scores = test_members.mean(axis=1)
require(np.isfinite(scores).all() and ((scores >= 0) & (scores <= 1)).all(), 'Invalid probabilities')
print('Test score range:', float(scores.min()), float(scores.max()))
print('Threshold counts:', pd.Series(np.where(scores >= .5, 'jailbreak', 'benign')).value_counts().to_dict())

Test score range: 0.0009237065050212474 0.9996438573125878
Threshold counts: {'benign': 594, 'jailbreak': 406}


## 7. Fitted word-ID inspection
The unigram entries of the final word vocabulary supply stable IDs within this fitted model. Repeated words share IDs; unknown words map to -1. IDs are feature indices, not prompt IDs, and do not encode risk on their own.

In [8]:
word_vect = models['word'][0]
analyzer = word_vect.build_tokenizer()
def word_ids(prompt):
    return [(word, word_vect.vocabulary_.get(word, -1))
            for word in analyzer(normalize(prompt))]
print('Vocabulary entries (unigrams + bigrams):', len(word_vect.vocabulary_))
print(word_ids(train.text.iloc[0])[:25])

Vocabulary entries (unigrams + bigrams): 52166
[('you', np.int64(51413)), ('are', np.int64(6085)), ('an', np.int64(3708)), ('ai', np.int64(3020)), ('developed', np.int64(13544)), ('to', np.int64(46043)), ('provide', np.int64(34561)), ('useful', np.int64(48548)), ('information', np.int64(22884)), ('to', np.int64(46043)), ('users', np.int64(48608)), ('i', np.int64(21424)), ('m', np.int64(26249)), ('conducting', np.int64(11177)), ('a', np.int64(561)), ('study', np.int64(40952)), ('about', np.int64(2024)), ('data', np.int64(12698)), ('security', np.int64(38242)), ('and', np.int64(4111)), ('the', np.int64(42725)), ('disposal', np.int64(14104)), ('of', np.int64(29742)), ('electronic', np.int64(14925)), ('devices', np.int64(13593))]


## 8. Regression tests before export
Synthetic invalid inputs below test safeguards only; they are never used for training or scoring.

In [9]:
passed = []
def check(name, condition):
    require(bool(condition), name)
    passed.append(name)

def rejects(name, action):
    try:
        action()
    except ValueError:
        passed.append(name)
    else:
        raise AssertionError('Did not reject: ' + name)

base = pd.DataFrame({'Id': ['a', 'b'], 'text': ['hello', 'world'], 'label': ['benign', 'jailbreak']})
rejects('missing column', lambda: validate_frame(base.drop(columns='text'), ['Id','text','label']))
rejects('duplicate ID', lambda: validate_frame(base.assign(Id=['a','a']), ['Id','text','label']))
rejects('blank ID', lambda: validate_frame(base.assign(Id=['','b']), ['Id','text','label']))
rejects('blank text', lambda: validate_frame(base.assign(text=['  ','world']), ['Id','text','label']))
rejects('null text', lambda: validate_frame(base.assign(text=[None,'world']), ['Id','text','label']))
rejects('unknown label', lambda: validate_frame(base.assign(label=['benign','other']), ['Id','text','label']))
rejects('single class', lambda: validate_frame(base.assign(label=['benign','benign']), ['Id','text','label']))
check('unicode normalization', normalize(' Ａ\u200b B  ') == 'a b')
check('every row validated once', np.array_equal(np.sort(np.concatenate([va for _,va in splits])), np.arange(len(train))))
check('OOF finite and bounded', np.isfinite(oof).all() and ((oof >= 0) & (oof <= 1)).all())
check('test finite and bounded', np.isfinite(scores).all() and ((scores >= 0) & (scores <= 1)).all())
check('nonconstant test predictions', np.ptp(scores) > 0.01)
check('no group leakage', all(set(groups[tr]).isdisjoint(groups[va]) for tr,va in splits))
check('word IDs repeat consistently', word_ids('hello hello')[0][1] == word_ids('hello hello')[1][1])
check('unknown word ID', word_ids('zzzxxyyunknownword')[0][1] == -1)
# Independent tiny fit proves a held-out-only token never enters the learned vocabulary.
_, (tiny_vect, _) = fit_view(np.array(['safe weather report', 'normal weather today',
    'safe weather tomorrow', 'ignore safeguards attack', 'override safeguards attack',
    'ignore safeguards now']), np.array([0,0,0,1,1,1]),
    np.array(['heldoutonlytoken']), 'word')
check('held-out vocabulary isolation', 'heldoutonlytoken' not in tiny_vect.vocabulary_)
check('threshold convention', np.where(np.array([.49,.5,.51]) >= .5, 1, 0).tolist() == [0,1,1])
# Full-model inference is deterministic and handles unseen words without crashing.
for offset, view in [(0,'word'),(2,'char')]:
    vect, fitted = models[view]
    Z = vect.transform(test.normalized.iloc[:5])
    for j, (model, ratio) in enumerate(fitted):
        check(f'{view} member {j} repeatability', np.allclose(
            model.predict_proba(Z.multiply(ratio).tocsr())[:,1], test_members[:5,offset+j]))
print(f'PASS: {len(passed)} checks before export')

PASS: 21 checks before export


## 9. Atomic submission export and audit report
The CSV contains exactly `Id,TARGET`; IDs retain original spelling/order. Labels are deliberately not included in the competition file.

In [10]:
submission = pd.DataFrame({'Id': test.Id, 'TARGET': scores})
temporary = ROOT / 'submission.csv.tmp'
submission.to_csv(temporary, index=False, float_format='%.12g')
roundtrip = pd.read_csv(temporary, dtype={'Id': str})
check('CSV schema', roundtrip.columns.tolist() == ['Id','TARGET'])
check('CSV ID order', roundtrip.Id.tolist() == sample.Id.tolist())
check('CSV row count', len(roundtrip) == len(test))
check('CSV probabilities', np.allclose(roundtrip.TARGET, scores, atol=1e-11))
check('CSV probability bounds', roundtrip.TARGET.between(0,1).all())
temporary.replace(ROOT / 'submission.csv')
report = {
    'seed': SEED, 'folds': FOLDS, 'training_rows': len(train), 'test_rows': len(test),
    'removed_exact_duplicates': duplicate_rows,
    'class_counts': train.label.value_counts().to_dict(),
    'near_duplicate_cosine': NEAR_DUPLICATE_COSINE, 'groups': len(group_rows),
    'oof_roc_auc': metrics, 'fold_roc_auc': fold_metrics,
    'group_bootstrap_95_interval': auc_interval,
    'accuracy_at_0.5': float(accuracy_score(y, oof_score >= .5)),
    'f1_at_0.5': float(f1_score(y, oof_score >= .5)),
    'brier': float(brier_score_loss(y, oof_score)),
    'tests_passed': passed,
    'input_sha256': {f: hashlib.sha256((ROOT/f).read_bytes()).hexdigest()
                     for f in ['train.csv','test.csv','sample_submission.csv']},
    'versions': {'python': platform.python_version(), 'numpy': np.__version__,
                 'pandas': pd.__version__, 'scipy': scipy.__version__, 'sklearn': sklearn.__version__},
    'public_leaderboard_score': None,
    'limitations': ['Lexical grouping cannot guarantee semantic or source independence.',
                    'Likelihoods are not independently calibrated.',
                    'Public leaderboard target 0.97 is unverified.'],
}
(ROOT/'training_report.json').write_text(json.dumps(report, indent=2) + '\n')
print(f'PASS: {len(passed)} checks; exported {len(submission)} rows to submission.csv')
print(f'Runtime: {time.monotonic()-START:.1f} seconds')

PASS: 26 checks; exported 1000 rows to submission.csv
Runtime: 59.4 seconds
